# Segment membrane and membrane cavities for more accurate measurement using ML

TODO:

- [x] select only the data containing the membrane and CL
- [ ] do the other stuff

In [17]:
# modules
import os
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import dask
import dask.array
from scipy import ndimage
from skimage import filters, feature, io
from skimage.morphology import disk,ball
import sys
from itertools import combinations_with_replacement
import pickle

from dask.distributed import Client, LocalCluster

# Workaround that dask does not kill workers while they are busy fetching data: https://dask.discourse.group/t/dask-workers-killed-because-of-heartbeat-fail/856
# maybe this helps: https://www.youtube.com/watch?v=vF2VItVU5zg?
from dask import config as cfg
cfg.set({'distributed.scheduler.worker-ttl': None})

# get the ML functions, TODO: make a library once it works
pytrainpath = '/mpc/homes/fische_r/lib/co2ely-tomcat'
cwd = os.getcwd()
os.chdir(pytrainpath)
from filter_functions import image_filter
import training_functions as tfs
from training_functions import train_segmentation
from segmentation import segmentation
os.chdir(cwd)

#paths
import socket
import subprocess
host = socket.gethostname()
if host == 'mpc2959.psi.ch':
    gitpath = '/mpc/homes/fische_r/lib/co2ely-tomcat'
    toppath = '/mpc/homes/fische_r/NAS/DASCOELY'
    toppathSSD = '/mnt/SSD/fische_r/COELY'
    temppath = '/mnt/SSD/fische_r/tmp/joblib_tmp'
    training_path = '/mpc/homes/fische_r/NAS/DASCOELY/processing/04_membrane_ML/'
else:
    print('currently only host mpc2959.psi.ch supported'
    
path_02_4D = os.path.join(toppathSSD, '02_registered_3p1D') #h5 with registered data

# fetch githash
cwd = os.getcwd()
os.chdir(gitpath)
git_sha = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip()
githash = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
os.chdir(cwd)

### functionalities for interactive training

In [ ]:
from ipywidgets import Image
from ipywidgets import ColorPicker, IntSlider, link, AppLayout, HBox
from ipycanvas import  hold_canvas,  MultiCanvas #RoughCanvas,Canvas,

def on_mouse_down(x, y):
    global drawing
    global position
    global shape
    drawing = True
    position = (x, y)
    shape = [position]

def on_mouse_move(x, y):
    global drawing
    global position
    global shape
    if not drawing:
        return
    with hold_canvas():
        canvas.stroke_line(position[0], position[1], x, y)
        position = (x, y)
    shape.append(position)

def on_mouse_up(x, y):
    global drawing
    global positiondu
    global shape
    drawing = False
    with hold_canvas():
        canvas.stroke_line(position[0], position[1], x, y)
        canvas.fill_polygon(shape)
    shape = []
    
def display_feature(i, TS):
    print('selected '+TS.feature_names[i])
    im = TS.current_feat_stack[:,:,i]
    im8 = im-im.min()
    im8 = im8/im8.max()*255
    return im8

### fire up dask

In [ ]:
tempfolder = '/mnt/SSD/fische_r/tmp'
dask.config.config['temporary-directory'] = tempfolder

cluster = LocalCluster(dashboard_address=':35000', memory_limit = '160GB', n_workers=6, threads_per_worker=20) 
client = Client(cluster)
print('Dashboard at '+cluster.dashboard_link)

## Test ROI selection

In [3]:
# sample = '8x'

In [4]:
# # file = '01_'+sample+'_rotated_3p1D.nc'
# file = '02_'+sample+'_registered_3p1D.nc'
# imagepath = os.path.join(path_02_4D, file)
# data = xr.open_dataset(imagepath)
# images = [im for im in data.keys() if im[3:7] == 'imag']
# images.sort()

In [5]:
# im300 = data[images[4]][:,:,300].data

In [6]:
# plt.imshow(im300[:,260:-260])

In [7]:
# data.close()

In [8]:
## [:, 250:-250, 50:-50] appears to be a good ROI for all samples

In [9]:
# crop = 260
# samplesall = ['1', '3II', '3III', '4', '4II', '5II', '5', '6', '7x', '8x']
# for sample in samplesall:
#     file = '02_'+sample+'_registered_3p1D.nc'
#     imagepath = os.path.join(path_02_4D, file)
#     data = xr.open_dataset(imagepath)
#     images = [im for im in data.keys() if im[3:7] == 'imag']
#     images.sort()
#     im300 = data[images[4]][:,:,300].data
#     im0 = data[images[4]][:,:,50].data
#     imfin = data[images[4]][:,:,-50].data
#     data.close()
#     fig, axs = plt.subplots(1,3)
#     axs[0].imshow(im300[:,crop:-crop])
#     axs[1].imshow(im0[:,crop:-crop])
#     axs[2].imshow(imfin[:,crop:-crop])
#     plt.title(sample +' '+ images[4])
    

## Data preparation

### create dask array

In [10]:
sample = '3II'
file = '02_'+sample+'_registered_3p1D.nc'
imagepath = os.path.join(path_02_4D, file)
data = xr.open_dataset(imagepath)
images = [im for im in data.keys() if im[3:7] == 'imag']
images.sort()

In [11]:
t_utc = data['t_utc'].data
time = data['time'].data

In [12]:
shp = data[images[4]][:, 250:-250, 50:-50].shape
shp = shp + (len(images),)

In [13]:
im = np.zeros(shp, dtype = np.uint16)

In [14]:
for i in range(shp[-1]):
    im[...,i] = data[images[i]][:, 250:-250, 50:-50].data

In [15]:
im.size/1024**3

14.343198671936989

In [18]:
da = dask.array.from_array(im, chunks = '30 MiB')

In [19]:
del im

In [20]:
da

dask.array<array, shape=(764, 167, 1916, 63), dtype=uint16, chunksize=(62, 62, 62, 62), chunktype=numpy.ndarray>

### get into image filter class

In [ ]:
IF = image_filter(chunksize = '30 MiB') #default feauture settings sigmas = [0,2, 4],
IF.data = da
IF.original_dataset = data


### prepare features

In [21]:
IF.prepare()

In [ ]:
IF.stack_features()

In [ ]:
IF.feature_stack

In [ ]:
IF.make_xarray_nc()

## Training

### set up objects

In [ ]:
training_path = os.path.join(training_path, sample)
if not os.path.exists(training_path):
    os.mkdir(training_path)

In [ ]:
TS = train_segmentation(training_path=training_path)

In [ ]:
TS.import_lazy_feature_data(IF.result, IF.original_dataset)

### interactive training

#### check for existing training sets

In [ ]:
os.listdir(os.path.join(training_path, 'label_images'))

#### suggest a new training coordinate

In [ ]:
TS.suggest_training_set()

In [ ]:
c1 = 'y'
p1 = 137
c2 = 'time'
p2 = 32

In [ ]:
TS.load_training_set(c1, p1, c2, p2)

#### canvas for labeling

In [ ]:
alpha = 0.35
im8 = TS.current_im8

#trick: use gaussian_time_4_0 to label static phases (-10)
# im8 = display_feature(-10, TS)

resultim = TS.current_result
width = im8.shape[1]
height = im8.shape[0]
Mcanvas = MultiCanvas(4, width=width, height=height)
background = Mcanvas[0]
resultdisplay = Mcanvas[2]
truthdisplay = Mcanvas[1]
canvas = Mcanvas[3]
canvas.sync_image_data = True
drawing = False
position = None
shape = []
image_data = np.stack((im8, im8, im8), axis=2)
background.put_image_data(image_data, 0, 0)
resultdisplay.global_alpha = alpha
if np.any(resultim>0):
    result_data = np.stack((255*(resultim==0), 255*(resultim==1), 255*(resultim==2)), axis=2)
else:
    result_data = np.stack((0*resultim, 0*resultim, 0*resultim), axis=2)
resultdisplay.put_image_data(result_data, 0, 0)
canvas.on_mouse_down(on_mouse_down)
canvas.on_mouse_move(on_mouse_move)
canvas.on_mouse_up(on_mouse_up)
picker = ColorPicker(description="Color:", value="#ff0000")
slidealpha = IntSlider(description="Result overlay", value=0.15)
link((picker, "value"), (canvas, "stroke_style"))
link((picker, "value"), (canvas, "fill_style"))
HBox((Mcanvas, picker, slidealpha))

#### inspect labels and training progress

In [ ]:
fig, axes = plt.subplots(1,6, figsize=(20,10))
axes[0].imshow(TS.current_result, 'gray')
axes[1].imshow(TS.current_im8, 'gray')

# TS.current_diff_im = TS.current_im-TS.current_first_im
# TS.current_diff_im = TS.current_diff_im/TS.current_diff_im.max()*255
axes[2].imshow(-TS.current_diff_im,vmin=6e4)
# axes[3].imshow(m8old, 'gray')
axes[3].imshow(TS.current_first_im, 'gray')
axes[4].imshow(TS.current_truth)
if TS.current_computed:
    axes[5].imshow(TS.current_feat_stack[:,:,-10])
else:
    axes[5].imshow(TS.current_result, 'gray')

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

#### update training set if labels are ok

In [ ]:
label_set = canvas.get_image_data()

TS.current_truth[label_set[:,:,0]>0] = 1
TS.current_truth[label_set[:,:,1]>0] = 2
TS.current_truth[label_set[:,:,2]>0] = 4

imageio.imsave(TS.current_truthpath, TS.current_truth)

#### train!

In [ ]:
TS.train_slice()

#### go back until happy

#### check on training progress by plausible feature importance

In [ ]:
plt.figure(figsize=(16,9))
plt.stem(TS.feature_names, TS.clf.feature_importances_,'x')
plt.xticks(rotation=90)
plt.ylabel('importance') 

### when done, maybe save the classifier

In [ ]:
TS.pickle_classifier()
pickle.dump(TS.training_dict, open(os.path.join(TS.training_path, 'training_dict.p'),'wb'))